In [ ]:
import gmsh
from collections import defaultdict

gmsh.initialize()
gmsh.option.setString("Geometry.OCCTargetUnit", "M")
gmsh.model.add("cylinder")

# Load CAD
cad_file_path = "partitioned_cylinder.step"
gmsh.model.occ.importShapes(cad_file_path)
gmsh.model.occ.synchronize()

# Define physical groups
# inlet_tag = 1
# outlet_tag = 3
# volume_tag = 1
# Extract raw volumes
volumes = gmsh.model.occ.getEntities(3)
print(f"Extracted {len(volumes)} raw volumes from CAD.")

# Fragment volumes
print("Fragmenting volumes...")
gmsh.model.occ.fragment(volumes, [])
gmsh.model.occ.removeAllDuplicates()
gmsh.model.occ.synchronize()

top = 1
bottom = 3
slab = 2

top_inlet = 4
bottom_inlet = 14
top_outlet = 2
bottom_outlet = 11

surfaces = gmsh.model.getEntities(dim=2)
surface_tags = [s[1] for s in surfaces]

# Final volumes
final_volumes = gmsh.model.getEntities(3)
print(f"Final number of volumes: {len(final_volumes)}")
total_surfaces = [s[1] for s in gmsh.model.getEntities(2)]
print(f"Total number of surfaces: {len(total_surfaces)}")

# ##############################################
# Build volume→surface mapping
# ##############################################
volume_surfaces = {}

for dim, vol_tag in final_volumes:
    boundary = gmsh.model.getBoundary([(3, vol_tag)], oriented=False, combined=False)
    surf_tags = [s[1] for s in boundary if s[0] == 2]
    volume_surfaces[vol_tag] = surf_tags

# Count occurrences to find shared surfaces
surface_counts = defaultdict(int)
for surf_list in volume_surfaces.values():
    for s in surf_list:
        surface_counts[s] += 1

shared_surfaces = [s for s, c in surface_counts.items() if c > 1]
print("Shared surfaces:", shared_surfaces)

# gmsh.model.addPhysicalGroup(2, [inlet_tag], inlet_marker, name="inlet")
# gmsh.model.addPhysicalGroup(2, [outlet_tag], outlet_marker, name="outlet")
# gmsh.model.addPhysicalGroup(2, wall_tags, wall_marker, name="walls")
# gmsh.model.addPhysicalGroup(3, [volume_tag], fluid_marker, name="fluid")

# ----------------------------------------
# MESH REFINEMENT ON SURFACES
# ----------------------------------------

# Global mesh size
# gmsh.option.setNumber("Mesh.CharacteristicLengthMax", 1e-2)

# Generate mesh
gmsh.model.mesh.generate(3)
gmsh.write("first_mesh_cylinder.msh")

gmsh.fltk.run()
gmsh.finalize()

Info    :  - Label 'Shapes/5084644a-eff9-11f0-bec2-bac16f87c4a8/fluid_half_1/fluid_half_1' (3D)
Info    :  - Label 'Shapes/5084644a-eff9-11f0-bec2-bac16f87c4a8/solid_slab/solid_slab' (3D)
Info    :  - Label 'Shapes/5084644a-eff9-11f0-bec2-bac16f87c4a8/fluid_half_2/fluid_half_2' (3D)
Extracted 3 raw volumes from CAD.
Fragmenting volumes...
Final number of volumes: 3                                                                                                                      
Total number of surfaces: 16
Shared surfaces: [3, 13]
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 10%] Meshing curve 2 (Circle)
Info    : [ 10%] Meshing curve 3 (Line)
Info    : [ 20%] Meshing curve 4 (Circle)
Info    : [ 20%] Meshing curve 5 (Line)
Info    : [ 20%] Meshing curve 6 (Line)
Info    : [ 30%] Meshing curve 19 (Circle)
Info    : [ 30%] Meshing curve 20 (Line)
Info    : [ 30%] Meshing curve 21 (Line)
Info    : [ 40%] Meshing curve 22 (Line)
Info    : [ 40%] Meshing c

In [ ]:
import gmsh
from collections import defaultdict

print("GMSH Version:", gmsh.__version__)

gmsh.initialize()
gmsh.option.setString("Geometry.OCCTargetUnit", "M")
gmsh.model.add("partitioned_cylinder")

# -----------------------------
# USER PARAMETERS
# -----------------------------
cad_file_path = "partitioned_cylinder.step"
H = 200.0
tol = 1e-6

# Physical IDs
top = 1
slab = 2
bottom = 3

top_inlet = 4
top_outlet = 2
bottom_inlet = 14
bottom_outlet = 11

# -----------------------------
# IMPORT CAD
# -----------------------------
gmsh.model.occ.importShapes(cad_file_path)
gmsh.model.occ.synchronize()

# -----------------------------
# FRAGMENT (CRITICAL)
# -----------------------------
volumes = gmsh.model.occ.getEntities(3)
print(f"Raw volumes: {len(volumes)}")

gmsh.model.occ.fragment(volumes, [])
gmsh.model.occ.removeAllDuplicates()
gmsh.model.occ.synchronize()

final_volumes = gmsh.model.getEntities(3)
surfaces = gmsh.model.getEntities(2)

print(f"Final volumes: {len(final_volumes)}")
print(f"Total surfaces: {len(surfaces)}")

# -----------------------------
# BUILD volume → surface map
# -----------------------------
volume_surfaces = {}

for dim, vtag in final_volumes:
    bnd = gmsh.model.getBoundary([(3, vtag)], oriented=False, combined=False)
    volume_surfaces[vtag] = [s[1] for s in bnd if s[0] == 2]

# -----------------------------
# FIND SHARED SURFACES
# -----------------------------
surface_count = defaultdict(int)
for slist in volume_surfaces.values():
    for s in slist:
        surface_count[s] += 1

shared_surfaces = [s for s, c in surface_count.items() if c > 1]
print("Shared surfaces:", shared_surfaces)

# -----------------------------
# CLASSIFY VOLUMES
# -----------------------------
centroids = {
    v: gmsh.model.occ.getCenterOfMass(3, v)
    for _, v in final_volumes
}

top_vol = max(centroids, key=lambda v: centroids[v][1])
bottom_vol = min(centroids, key=lambda v: centroids[v][1])
slab_vol = list(set(centroids) - {top_vol, bottom_vol})[0]

gmsh.model.addPhysicalGroup(3, [top_vol], top, name="top_fluid")
gmsh.model.addPhysicalGroup(3, [slab_vol], slab, name="slab")
gmsh.model.addPhysicalGroup(3, [bottom_vol], bottom, name="bottom_fluid")

# -----------------------------
# CLASSIFY BOUNDARY SURFACES
# -----------------------------
zmin = -H / 2
zmax =  H / 2

for dim, s in surfaces:
    if s in shared_surfaces:
        continue  # internal interface

    x, y, z = gmsh.model.occ.getCenterOfMass(2, s)

    # TOP
    if y > 0:
        if abs(z - zmin) < tol:
            gmsh.model.addPhysicalGroup(2, [s], top_inlet, name="top_inlet")
        elif abs(z - zmax) < tol:
            gmsh.model.addPhysicalGroup(2, [s], top_outlet, name="top_outlet")

    # BOTTOM
    if y < 0:
        if abs(z - zmin) < tol:
            gmsh.model.addPhysicalGroup(2, [s], bottom_inlet, name="bottom_inlet")
        elif abs(z - zmax) < tol:
            gmsh.model.addPhysicalGroup(2, [s], bottom_outlet, name="bottom_outlet")

# -----------------------------
# OPTIONAL: WALLS
# -----------------------------
wall_surfaces = [
    s for _, s in surfaces
    if s not in shared_surfaces
]

gmsh.model.addPhysicalGroup(2, wall_surfaces, 99, name="walls")

# -----------------------------
# MESH
# -----------------------------
gmsh.option.setNumber("Mesh.MshFileVersion", 2.2)
gmsh.model.mesh.generate(3)
gmsh.write("partitioned_cylinder.msh")

gmsh.fltk.run()
gmsh.finalize()


GMSH Version: 4.15.0
Info    :  - Label 'Shapes/5084644a-eff9-11f0-bec2-bac16f87c4a8/fluid_half_1/fluid_half_1' (3D)
Info    :  - Label 'Shapes/5084644a-eff9-11f0-bec2-bac16f87c4a8/solid_slab/solid_slab' (3D)
Info    :  - Label 'Shapes/5084644a-eff9-11f0-bec2-bac16f87c4a8/fluid_half_2/fluid_half_2' (3D)
Raw volumes: 3
Final volumes: 3                                                                                                                                
Total surfaces: 16
Shared surfaces: [3, 13]
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 10%] Meshing curve 2 (Circle)
Info    : [ 10%] Meshing curve 3 (Line)
Info    : [ 20%] Meshing curve 4 (Circle)
Info    : [ 20%] Meshing curve 5 (Line)
Info    : [ 20%] Meshing curve 6 (Line)
Info    : [ 30%] Meshing curve 19 (Circle)
Info    : [ 30%] Meshing curve 20 (Line)
Info    : [ 30%] Meshing curve 21 (Line)
Info    : [ 40%] Meshing curve 22 (Line)
Info    : [ 40%] Meshing curve 23 (Circle)
Info    : [ 40